# Debug Individual Event Training

Run individual events through the Cox model pipeline to diagnose NaN issues.
Tests data quality and model fitting for specific events (death, metastasis).

In [ ]:
import sys
import os
import logging
import numpy as np
import pandas as pd

# Add project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'python_utils'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'python_scripts', 'model_training'))

from embed_surv_utils import run_base_CoxPH, run_grid_CoxPH_parallel
from slurm_array_utils import (
    DEFAULT_ALPHAS, DEFAULT_L1_RATIOS, MET_EVENTS,
    build_full_prediction_df, filter_event_rows,
)

# Enable logging so we can see warnings from cox_models
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('embed_surv_utils.cox_models')
logger.setLevel(logging.DEBUG)

## 1. Load data and inspect available events

In [ ]:
SCHEME = 'death_met'  # change to 'icd3', 'icd4', 'phecode' as needed

full_prediction_df, type_cols, embed_cols, events = build_full_prediction_df(SCHEME)
print(f"Scheme: {SCHEME}")
print(f"Shape: {full_prediction_df.shape}")
print(f"Events found ({len(events)}): {events}")
print(f"Embedding cols: {len(embed_cols)}")
print(f"Cancer type cols: {type_cols}")

## 2. Data quality diagnostics per event

Check NaN counts, negative times, event rates, and sample sizes for each event.

In [ ]:
# Events to debug - death + all met events present in this scheme
debug_events = [e for e in events if e == 'death' or e in MET_EVENTS]
if not debug_events:
    # Fallback: just use the first few events
    debug_events = events[:5]
print(f"Debug events: {debug_events}")

In [ ]:
diag_rows = []
for event in debug_events:
    tt_col = f'tt_{event}'
    tt = full_prediction_df[tt_col]
    ev = full_prediction_df[event]
    
    row = {
        'event': event,
        'n_total': len(tt),
        'tt_nan': tt.isna().sum(),
        'tt_negative': (tt < 0).sum(),
        'tt_zero': (tt == 0).sum(),
        'tt_positive': (tt > 0).sum(),
        'event_nan': ev.isna().sum(),
        'event_rate': ev.mean(),
        'n_events': ev.sum(),
        'tt_min': tt.min(),
        'tt_max': tt.max(),
        'tt_median': tt.median(),
    }
    diag_rows.append(row)

diag_df = pd.DataFrame(diag_rows)
diag_df

In [ ]:
# Check for NaN in embedding columns
embed_nan_per_col = full_prediction_df[embed_cols].isna().sum()
n_embed_nan = embed_nan_per_col.sum()
print(f"Total NaN values across {len(embed_cols)} embedding columns: {n_embed_nan}")
if n_embed_nan > 0:
    print("\nColumns with NaN:")
    print(embed_nan_per_col[embed_nan_per_col > 0])

# Check for NaN in base covariates
base_vars = ['GENDER', 'AGE_AT_TREATMENTSTART']
for col in base_vars + type_cols:
    n_nan = full_prediction_df[col].isna().sum()
    if n_nan > 0:
        print(f"  {col}: {n_nan} NaN values")

## 3. Test individual events through the pipeline

Run `run_base_CoxPH` and `run_grid_CoxPH_parallel` on individual events to see which ones fail and what errors are raised.

In [ ]:
base_vars = ['GENDER', 'AGE_AT_TREATMENTSTART']

def run_single_event_debug(event, scheme_df, type_cols, embed_cols, run_grid=True):
    """Run both base and grid models for a single event, printing diagnostics."""
    tt_col = f'tt_{event}'
    print(f"\n{'='*60}")
    print(f"EVENT: {event}")
    print(f"{'='*60}")
    
    # Filter rows
    event_df = filter_event_rows(scheme_df, event)
    print(f"Rows after filtering: {len(event_df)} (from {len(scheme_df)})")
    print(f"Event rate: {event_df[event].mean():.4f} ({int(event_df[event].sum())} events)")
    print(f"Time range: [{event_df[tt_col].min():.1f}, {event_df[tt_col].max():.1f}] days")
    
    if event_df.empty:
        print("  -> SKIPPED: no valid rows")
        return None, None
    
    # Check feature matrix for NaN/inf
    all_feature_cols = base_vars + type_cols + embed_cols
    X_check = event_df[all_feature_cols].to_numpy(dtype=np.float32)
    print(f"Feature matrix: {X_check.shape}")
    print(f"  NaN count: {np.isnan(X_check).sum()}")
    print(f"  Inf count: {np.isinf(X_check).sum()}")
    
    # Zero-variance columns
    stds = np.std(X_check, axis=0)
    zero_var_cols = [all_feature_cols[i] for i in range(len(stds)) if stds[i] == 0]
    if zero_var_cols:
        print(f"  Zero-variance columns ({len(zero_var_cols)}): {zero_var_cols[:10]}...")
    
    # Run baseline CoxPH
    print("\n--- Baseline CoxPH ---")
    try:
        base_results = run_base_CoxPH(
            event_df, base_vars + type_cols, ['AGE_AT_TREATMENTSTART'],
            event_col=event, tstop_col=tt_col,
        )
        print(base_results.to_string(index=False))
        has_nan = base_results[['mean_c_index', 'mean_auc(t)', 'mean_ibs']].isna().any().any()
        if has_nan:
            print("  ** WARNING: NaN in baseline results **")
    except Exception as e:
        print(f"  ** FAILED: {e} **")
        base_results = None
    
    # Run grid search (text embeddings)
    grid_test = None
    if run_grid:
        print("\n--- Grid CoxPH (text embeddings) ---")
        try:
            grid_test, grid_val, _ = run_grid_CoxPH_parallel(
                event_df, base_vars + type_cols,
                ['AGE_AT_TREATMENTSTART'] + embed_cols,
                embed_cols,
                DEFAULT_L1_RATIOS, DEFAULT_ALPHAS,
                event_col=event, tstop_col=tt_col,
                max_iter=1000, n_jobs=1,  # single-threaded for debugging
            )
            print("Test metrics:")
            print(grid_test.to_string(index=False))
            
            # Show best CV result
            valid_cv = grid_val.dropna(subset=['mean_auc(t)'])
            if not valid_cv.empty:
                best = valid_cv.sort_values('mean_auc(t)', ascending=False).iloc[0]
                print(f"Best CV: alpha={best['alpha']:.2e}, l1={best['l1_ratio']:.1f}, "
                      f"AUC(t)={best['mean_auc(t)']:.4f}, error_rate={best['error_rate']:.2f}")
            else:
                print("  ** WARNING: ALL CV results are NaN **")
            
            has_nan = grid_test.isna().any().any()
            if has_nan:
                print("  ** WARNING: NaN in grid search test results **")
        except Exception as e:
            print(f"  ** FAILED: {e} **")
    
    return base_results, grid_test

### 3a. Test death event

In [ ]:
if 'death' in events:
    base_death, grid_death = run_single_event_debug(
        'death', full_prediction_df, type_cols, embed_cols, run_grid=True
    )
else:
    print("'death' event not found in this scheme. Available events:", events[:10])

### 3b. Test metastasis events

In [ ]:
met_events_present = [e for e in events if e in MET_EVENTS]
print(f"Metastasis events in this scheme: {met_events_present}")

met_results = {}
for met_event in met_events_present:
    base_met, grid_met = run_single_event_debug(
        met_event, full_prediction_df, type_cols, embed_cols, run_grid=True
    )
    met_results[met_event] = {'base': base_met, 'grid': grid_met}

## 4. Summary of results across tested events

In [ ]:
summary_rows = []

# Collect all tested events
all_tested = {}
if 'death' in events:
    all_tested['death'] = {'base': base_death, 'grid': grid_death}
all_tested.update(met_results)

for event_name, results in all_tested.items():
    row = {'event': event_name}
    
    if results['base'] is not None:
        cv_row = results['base'][results['base']['eval_data'] == 'cv_data']
        test_row = results['base'][results['base']['eval_data'] == 'test_data']
        row['base_cv_auc'] = cv_row['mean_auc(t)'].values[0] if len(cv_row) else np.nan
        row['base_test_auc'] = test_row['mean_auc(t)'].values[0] if len(test_row) else np.nan
    else:
        row['base_cv_auc'] = 'FAILED'
        row['base_test_auc'] = 'FAILED'
    
    if results['grid'] is not None:
        row['grid_test_auc'] = results['grid']['mean_auc(t)'].values[0]
        row['grid_test_cindex'] = results['grid']['mean_c_index'].values[0]
    else:
        row['grid_test_auc'] = 'FAILED'
        row['grid_test_cindex'] = 'FAILED'
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\nSummary of all tested events:")
summary_df

## 5. Quick baseline-only scan across all events

Run just the baseline CoxPH (fast, no embeddings) across all events to find which ones produce NaN.

In [ ]:
scan_rows = []
for event in events:
    tt_col = f'tt_{event}'
    event_df = filter_event_rows(full_prediction_df, event)
    if event_df.empty:
        scan_rows.append({'event': event, 'n_rows': 0, 'n_events': 0,
                          'cv_auc': 'EMPTY', 'test_auc': 'EMPTY'})
        continue
    
    try:
        base_res = run_base_CoxPH(
            event_df, base_vars + type_cols, ['AGE_AT_TREATMENTSTART'],
            event_col=event, tstop_col=tt_col,
        )
        cv_auc = base_res.loc[base_res['eval_data'] == 'cv_data', 'mean_auc(t)'].values[0]
        test_auc = base_res.loc[base_res['eval_data'] == 'test_data', 'mean_auc(t)'].values[0]
    except Exception as e:
        cv_auc = f'ERROR: {e}'
        test_auc = f'ERROR: {e}'
    
    scan_rows.append({
        'event': event,
        'n_rows': len(event_df),
        'n_events': int(event_df[event].sum()),
        'event_rate': f"{event_df[event].mean():.4f}",
        'cv_auc': cv_auc,
        'test_auc': test_auc,
    })

scan_df = pd.DataFrame(scan_rows)
print(f"Scanned {len(events)} events:")
scan_df

In [ ]:
# Show events that produced NaN
nan_events = scan_df[
    scan_df['cv_auc'].apply(lambda x: isinstance(x, float) and np.isnan(x))
    | scan_df['test_auc'].apply(lambda x: isinstance(x, float) and np.isnan(x))
    | scan_df['cv_auc'].apply(lambda x: isinstance(x, str) and x.startswith('ERROR'))
]
print(f"\nEvents with NaN or errors ({len(nan_events)}/{len(events)}):")
nan_events